<a href="https://colab.research.google.com/github/dominicwooldridge/GSB-S544/blob/Practice-Activities/PA_7_1_Cross_Validation_and_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


In [ ]:
ames = pd.read_csv('AmesHousing.csv')
ames.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [ ]:
ames.columns

Index(['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area',
       'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities',
       'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Overall Qual',
       'Overall Cond', 'Year Built', 'Year Remod/Add', 'Roof Style',
       'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type',
       'Mas Vnr Area', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual',
       'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1',
       'BsmtFin Type 2', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
       'Heating', 'Heating QC', 'Central Air', 'Electrical', '1st Flr SF',
       '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath',
       'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr',
       'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Functional',
       'Fireplaces', 'Fireplace Qu', 'Garage Type', 'Garage Yr Blt',
      

In [ ]:
target = "SalePrice"
num_cols_basic = ["Gr Liv Area", "TotRms AbvGrd"]
cat_cols = ["Bldg Type"]

X = ames[num_cols_basic + cat_cols]
y = ames[target]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
num_basic = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

cat_basic = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])


#1: Size + Rooms

In [ ]:
pre_1 = ColumnTransformer([
    ("num", num_basic, num_cols_basic),
], remainder="drop")
pipe_1 = Pipeline([
    ("pre", pre_1),
    ("model", LinearRegression())
])

# 2: Size + rooms + building type


In [ ]:
pre_2 = ColumnTransformer([
    ("num", num_basic, num_cols_basic),
    ("cat", cat_basic, cat_cols),
], remainder="drop")
pipe_2 = Pipeline([
    ("pre", pre_2),
    ("model", LinearRegression())
])

# 3: size + building type + their interaction

In [ ]:
pre_3_base = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), ["Gr Liv Area"]),
    ("cat", cat_basic, cat_cols),
], remainder="drop")
pipe_3 = Pipeline([
    ("pre_base", pre_3_base),
    ("interact", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ("model", LinearRegression())
])

# 4: 5-degree poly on size & rooms + building type


In [ ]:
num_poly5 = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=5, include_bias=False)),
    ("scale", StandardScaler()),
])
pre_4 = ColumnTransformer([
    ("num_poly5", num_poly5, num_cols_basic),
    ("cat", cat_basic, cat_cols),
], remainder="drop")
pipe_4 = Pipeline([
    ("pre", pre_4),
    ("model", LinearRegression())
])

# Fit and evaluate

In [ ]:
models = {
    "Model 1: size + rooms": pipe_1,
    "Model 2: size + rooms + type": pipe_2,
    "Model 3: size + type + interaction(size×type)": pipe_3,
    "Model 4: poly5(size, rooms) + type": pipe_4,
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results[name] = rmse(y_test, preds)

# Report:

In [ ]:
for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name}: RMSE = {score:,.2f}")

best = min(results, key=results.get)
print("\nBest model:", best)

Model 3: size + type + interaction(size×type): RMSE = 58,276.73
Model 2: size + rooms + type: RMSE = 59,589.20
Model 4: poly5(size, rooms) + type: RMSE = 61,791.59
Model 1: size + rooms: RMSE = 61,928.54

Best model: Model 3: size + type + interaction(size×type)


Model 3 perfomed best.

# Using cross_val_score:

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np


def rmse_cv(pipe, X, y, cv=5):
    neg_mse = cross_val_score(pipe, X, y,
                              scoring="neg_mean_squared_error",
                              cv=cv)
    rmse_scores = np.sqrt(-neg_mse)
    return rmse_scores.mean()

cv_results = {}
for name, pipe in models.items():
    cv_results[name] = rmse_cv(pipe, X, y, cv=5)


for name, score in sorted(cv_results.items(), key=lambda x: x[1]):
    print(f"{name}: Cross-validated RMSE = {score:,.2f}")

best_cv = min(cv_results, key=cv_results.get)
print("\nBest cross-validated model:", best_cv)


Model 3: size + type + interaction(size×type): Cross-validated RMSE = 53,430.92
Model 2: size + rooms + type: Cross-validated RMSE = 54,168.08
Model 1: size + rooms: Cross-validated RMSE = 55,806.33
Model 4: poly5(size, rooms) + type: Cross-validated RMSE = 70,854.54

Best cross-validated model: Model 3: size + type + interaction(size×type)


With cross validation, Model 3 still performs the best.

# Tuning:

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
import numpy as np

# --- Columns ---
num_cols = ["Gr Liv Area", "TotRms AbvGrd"]
cat_cols = ["Bldg Type"]

# --- Preprocessing for numeric & categorical parts ---
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

# --- Column transformer combining numeric + categorical ---
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# --- Main model pipeline ---
pipe = Pipeline([
    ("pre", preprocessor),
    ("poly", PolynomialFeatures(include_bias=False)),
    ("model", LinearRegression())
])

# --- Grid of polynomial degrees ---
param_grid = {
    "poly__degree": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
}

# --- GridSearchCV across both numeric variables
grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5
)

grid.fit(X, y)

# --- Get results ---
best_degree = grid.best_params_["poly__degree"]
best_rmse = np.sqrt(-grid.best_score_)

print(f"Best polynomial degree: {best_degree}")
print(f"Cross-validated RMSE: {best_rmse:,.2f}")


KeyboardInterrupt: 